## SchNetPack

- Source: https://github.com/atomistic-machine-learning/schnetpack
- Citation: Schütt, K. T., Sauceda, H. E., Kindermans, P.-J., Tkatchenko, A. & Müller, K.-R. SchNet – A deep learning architecture for molecules and materials. *J. Chem. Phys.* **148**, 241722 (2018).

Setup: download the repository and copy its `src/schnetpack` folder to the root of this repository. Install the dependencies of SchNetPack with pip:

```
pip install ase matscipy vesin fasteners dirsync torch-ema petname progressbar "hydra-core>=1.1.0" hydra-colorlog tensorboardX lightning torchmetrics rich tqdm
```

In [ ]:
import gc
import os
import warnings

import numpy as np
import pandas as pd
import schnetpack as spk
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from schnetpack import properties
from schnetpack.data.loader import _atoms_collate_fn
from schnetpack.transform import TorchNeighborList
from torch import nn
from torch.utils.data import DataLoader, Dataset

from benchmarks import (
    BestEpochTracker,
    LightningModule,
    MultiTargetScaler,
    PrintEpochMetricsCallback,
)

warnings.filterwarnings("ignore", message=".*does not have many workers.*")

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| schnetpack", spk.__version__)


k_folds = 5
FOLDS = range(1, 6)

target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]

TAG = "schnet_1.1M-conformer"
model_path_template = "../models/" + TAG + "-[PAC5-4]-fold_{fold}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = TAG + "-[PAC5-4]-fold_{fold}"

SPLIT_DIR = model_dataset_dir + "/PAC5-4"

N_INTERACTIONS = 3  # SchNetPack default is 6
N_RBF = 50
CUTOFF = 5.0  # Angstrom (SchNetPack's QM9 value)
N_ATOM_BASIS = 281  # Overwritten by parameter search below
N_FILTERS = None  # Equal to n_atom_basis
FFN_LAYERS = 2  # Atomwise default

BATCH_SIZE = 256

MAX_EPOCHS = 500
ES_PATIENCE = 50
LR_PATIENCE = 25
MIN_LR = 1e-6
LR = 1e-3
NUM_WORKERS = 12

# Built by prep_3d_geometry_cache.py
_geom = np.load(SPLIT_DIR + "/geom_openff.npz")
GEOM_Z, GEOM_R, GEOM_PTR = _geom["Z"], _geom["R"], _geom["ptr"]
_gidx = pd.read_csv(SPLIT_DIR + "/geom_openff_index.csv")
SMILES_TO_ROW = dict(zip(_gidx["smiles"].values, _gidx["row"].values))
print(
    f"Geometry cache: {len(GEOM_PTR) - 1} molecules, {len(GEOM_Z)} atoms, "
    f"{np.diff(GEOM_PTR).mean():.1f} atoms/molecule (H included)"
)
del _geom, _gidx


class PlateauLightningModule(LightningModule):
    """LightningModule from benchmarks with ReduceLROnPlateau instead of OneCycleLR."""

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=LR_PATIENCE, min_lr=MIN_LR
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }


class PAC5GeometryDataset(Dataset):
    """PAC5 molecules as SchNetPack input dictionaries."""

    def __init__(self, smiles, scaled_targets, cutoff=CUTOFF):
        rows, keep = [], []
        for i, s in enumerate(smiles):
            r = SMILES_TO_ROW.get(s)
            if r is not None:
                rows.append(r)
                keep.append(i)
        self.rows = np.asarray(rows, dtype=np.int64)
        self.y = torch.as_tensor(np.asarray(scaled_targets)[keep], dtype=torch.float32)
        self.n_dropped = len(smiles) - len(self.rows)
        self.neighbor_list = TorchNeighborList(cutoff=cutoff)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        s, e = GEOM_PTR[row], GEOM_PTR[row + 1]
        Z = torch.from_numpy(GEOM_Z[s:e].astype(np.int64))
        R = torch.from_numpy(np.ascontiguousarray(GEOM_R[s:e]))

        inputs = {
            properties.Z: Z,
            properties.R: R,
            properties.cell: torch.zeros(3, 3, dtype=torch.float32),
            properties.pbc: torch.zeros(3, dtype=torch.bool),
            properties.n_atoms: torch.tensor([Z.shape[0]], dtype=torch.long),
        }
        inputs = self.neighbor_list(inputs)
        return inputs, self.y[index]


def make_collate(scaler=None):
    """Wraps _atoms_collate_fn so the batch is (inputs, y)"""

    def _collate(samples):
        graphs = [s[0] for s in samples]
        y = torch.stack([s[1] for s in samples])
        return _atoms_collate_fn(graphs), y

    return _collate


class SchNetMultiTask(nn.Module):
    """SchNet. Mean readout for intensive targets. Sum readout for the extensive targets."""

    def __init__(
        self,
        property_types,
        n_atom_basis=N_ATOM_BASIS,
        n_interactions=N_INTERACTIONS,
        n_filters=N_FILTERS,
        n_rbf=N_RBF,
        cutoff=CUTOFF,
        ffn_layers=FFN_LAYERS,
        n_hidden=None,
    ):
        super().__init__()
        self.property_types = list(property_types)
        self.n_int = self.property_types.count("intensive")
        self.n_ext = self.property_types.count("extensive")

        self.pairwise = spk.atomistic.PairwiseDistances()
        self.representation = spk.representation.SchNet(
            n_atom_basis=n_atom_basis,
            n_interactions=n_interactions,
            n_filters=n_filters,
            radial_basis=spk.nn.GaussianRBF(n_rbf=n_rbf, cutoff=cutoff),
            cutoff_fn=spk.nn.CosineCutoff(cutoff),
        )

        # Both heads sum; intensive head is divided by atom count below
        # Atomwise aggregation_mode='avg' broadcasts incorrectly for n_out > 1
        if self.n_int:
            self.int_head = spk.atomistic.Atomwise(
                n_in=n_atom_basis,
                n_out=self.n_int,
                n_hidden=n_hidden,
                n_layers=ffn_layers,
                aggregation_mode="sum",
                output_key="y_int",
            )
        if self.n_ext:
            self.ext_head = spk.atomistic.Atomwise(
                n_in=n_atom_basis,
                n_out=self.n_ext,
                n_hidden=n_hidden,
                n_layers=ffn_layers,
                aggregation_mode="sum",
                output_key="y_ext",
            )

    def forward(self, inputs):
        inputs = self.pairwise(inputs)
        inputs = self.representation(inputs)
        n_mol = int(inputs[properties.idx_m][-1]) + 1
        n_atoms = (
            inputs[properties.n_atoms]
            .view(n_mol, 1)
            .to(inputs["scalar_representation"].dtype)
        )

        int_pred = None
        ext_pred = None
        if self.n_int:
            int_pred = self.int_head(inputs)["y_int"].view(n_mol, self.n_int) / n_atoms
        if self.n_ext:
            ext_pred = self.ext_head(inputs)["y_ext"].view(n_mol, self.n_ext)

        out, i, e = [], 0, 0
        for p in self.property_types:
            if p == "intensive":
                if int_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for intensive properties."
                    )
                out.append(int_pred[:, i : i + 1])
                i += 1
            else:
                if ext_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for extensive properties."
                    )
                out.append(ext_pred[:, e : e + 1])
                e += 1
        return torch.cat(out, dim=1)

In [ ]:
TARGET_PARAMS = 1_100_000


def count_params(n_atom_basis):
    m = SchNetMultiTask(property_types, n_atom_basis=n_atom_basis)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    del m
    return n


print("======[ Parameter count by n_atom_basis ]======")
best, best_gap = None, None
for nb in range(96, 385):
    n = count_params(nb)
    gap = abs(n - TARGET_PARAMS)
    if best is None or best_gap is None or gap < best_gap:
        best, best_gap = nb, gap
    if nb % 8 == 0 and 0.9e6 < n < 1.3e6:
        print(f" + n_atom_basis {nb}: {n:,} parameters")

N_ATOM_BASIS = best
print(
    f"\nChosen n_atom_basis = {N_ATOM_BASIS} -> {count_params(N_ATOM_BASIS):,} parameters "
    f"(target {TARGET_PARAMS:,}, depth {N_INTERACTIONS})"
)

_df = (
    pd.read_csv(SPLIT_DIR + "/fold_1_val.csv").dropna(subset=target_properties).head(64)
)
_sc = MultiTargetScaler()
_sc.fit(_df[target_properties].values.astype(np.float32))
_ds = PAC5GeometryDataset(
    _df["smiles"].values,
    _sc.transform(_df[target_properties].values.astype(np.float32)),
)
print(f"Smoke set: {len(_ds)} molecules, {_ds.n_dropped} without a cached geometry")
_dl = DataLoader(
    _ds, batch_size=32, shuffle=False, collate_fn=make_collate(), num_workers=0
)
_batch = next(iter(_dl))
_m = SchNetMultiTask(property_types, n_atom_basis=N_ATOM_BASIS)
with torch.no_grad():
    _out = _m(_batch[0])
print("Batch keys:", sorted(_batch[0].keys()))
print(
    "Edges:",
    _batch[0][properties.idx_i].shape[0],
    "| atoms:",
    _batch[0][properties.Z].shape[0],
)
print("Output:", tuple(_out.shape), "expected", (32, len(target_properties)))
assert _out.shape == (32, len(target_properties)) and torch.isfinite(_out).all()
del _df, _sc, _ds, _dl, _batch, _m, _out
gc.collect()

In [ ]:
fold_results = []
best_epochs = []
stopped_epochs = []

for fold in FOLDS:
    if fold in [3, 4, 5]:
        pass
    else:
        continue
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_train.csv").dropna(
        subset=target_properties
    )
    val_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_val.csv").dropna(
        subset=target_properties
    )

    train_targets = train_df[target_properties].values.astype(np.float32)
    val_targets = val_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    train_dataset = PAC5GeometryDataset(
        train_df["smiles"].values, scaler.transform(train_targets)
    )
    val_dataset = PAC5GeometryDataset(
        val_df["smiles"].values, scaler.transform(val_targets)
    )
    print(
        f"Train {len(train_dataset)} (dropped {train_dataset.n_dropped}) | "
        f"Val {len(val_dataset)} (dropped {val_dataset.n_dropped})"
    )

    coll = make_collate()
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )

    model = SchNetMultiTask(property_types, n_atom_basis=N_ATOM_BASIS)
    print(
        f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    )
    lightning_model = PlateauLightningModule(model, target_properties, scaler, lr=LR)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=ES_PATIENCE,
        min_delta=0.0,
        check_on_train_epoch_end=False,
        verbose=False,
    )
    best_epoch_tracker = BestEpochTracker()

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[
            checkpoint_callback,
            early_stop,
            best_epoch_tracker,
            PrintEpochMetricsCallback(target_properties),
        ],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    best_epoch, last_epoch = best_epoch_tracker.best_epoch, best_epoch_tracker.last_epoch
    if best_epoch is None:
        raise RuntimeError("The best-epoch tracker did not record a validation epoch.")
    best_epochs.append(best_epoch)
    stopped_epochs.append(last_epoch)
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    print(f" + Best epoch {best_epoch} of {last_epoch} run (ceiling {MAX_EPOCHS})")
    fold_results.append(best_val_loss)

    best_lightning_model = PlateauLightningModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_df, val_df, train_dataset, val_dataset, train_loader, val_loader
    del (
        model,
        lightning_model,
        best_lightning_model,
        trainer,
        checkpoint_callback,
        logger,
        scaler,
    )
    del early_stop, best_epoch_tracker

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"\nBest-validation epoch per fold: {best_epochs} (ran {stopped_epochs})")
print(
    f"Latest best-validation epoch: {max(best_epochs)} of a {MAX_EPOCHS}-epoch ceiling"
)

In [ ]:
metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in FOLDS:
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_train.csv").dropna(
        subset=target_properties
    )
    test_df = pd.read_csv(SPLIT_DIR + f"/fold_{fold}_test.csv").dropna(
        subset=target_properties
    )

    train_targets = train_df[target_properties].values.astype(np.float32)
    test_targets = test_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    train_dset = PAC5GeometryDataset(
        train_df["smiles"].values, scaler.transform(train_targets)
    )
    test_dset = PAC5GeometryDataset(
        test_df["smiles"].values, scaler.transform(test_targets)
    )

    coll = make_collate()
    train_loader = DataLoader(
        train_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )
    test_loader = DataLoader(
        test_dset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=coll,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=False,
    )

    model = SchNetMultiTask(property_types, n_atom_basis=N_ATOM_BASIS)
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = PlateauLightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train set ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test set ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    del model, lightning_model, trainer, scaler, train_results, test_results
    del train_loader, test_loader, train_dset, test_dset, train_df, test_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"======[ Cross-validation results: {TAG} ]======")

ddof = 1 if len(FOLDS) > 1 else 0
for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=ddof),
        )
        for m in metric_keys
    }
    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=ddof),
        )
        for m in metric_keys
    }
    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )